# 👑 ALICE DADANG SOVEREIGN — QLoRA TRAINER & GGUF EXPORT
**Arsitek & Pencipta**: Commander Dadang Wahyuono (Maestro Dadang)
**Fokus**: CME Order Flow + Storyline v4.2 + Valid Close Bar + Ahli Office & Financial Analytics
**Base Model**: `huihui-ai/Huihui-Qwen3-VL-8B-Instruct-abliterated` (100% Uncensored + Vision)
**Dataset**: `dadangwahyuono/alice-sovereign-doctrine-sft` (36,189 sampel)

In [ ]:
# [CELL 1] Cek Status Hardware & GPU Kaggle
import os
import torch

print("=" * 72)
print("👑 ALICE DADANG SOVEREIGN — INITIALIZING HARDWARE ENVIRONMENT")
print("=" * 72)
!nvidia-smi
print(f"\nCUDA Available: {torch.cuda.is_available()} | GPU Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU[{i}]: {props.name} ({props.total_memory / (1024**3):.2f} GB)")

In [ ]:
# [CELL 2] Install Dependensi Modern & Ekosistem GGUF
!pip install -q -U "transformers>=4.57.1" "peft>=0.14.0" "bitsandbytes>=0.45.0" "accelerate>=1.2.0" "datasets>=3.0.0" "huggingface_hub>=0.27.0" sentencepiece protobuf gguf
print("✅ Seluruh pustaka eksekusi berhasil terpasang.")

In [ ]:
# [CELL 3] Verifikasi & Download Dataset Doktrin SFT (Colab / Kaggle)
import os
from pathlib import Path

# Otomatis deteksi path dataset di Colab, Kaggle, atau lokal
DATASET_PATH = None
possible_paths = [
    Path("/content/alice_doctrine_combined.jsonl"),
    *(list(Path("/kaggle/input").rglob("alice_doctrine_combined.jsonl")) if Path("/kaggle/input").exists() else []),
    Path("./alice_doctrine_combined.jsonl")
]
for p in possible_paths:
    if p.exists() and p.stat().st_size > 1000000:
        DATASET_PATH = str(p)
        break

if not DATASET_PATH:
    print("Mengunduh dataset doktrin langsung dari server kedaulatan Miix Tower...")
    target_path = Path("/content/alice_doctrine_combined.jsonl") if Path("/content").exists() else Path("./alice_doctrine_combined.jsonl")
    !curl -L -s -o {target_path} https://trade.dadangchatai.com/static/alice_doctrine_combined.jsonl
    DATASET_PATH = str(target_path)

sz_mb = os.path.getsize(DATASET_PATH) / (1024 * 1024)
print(f"✅ Dataset siap dan terverifikasi: {DATASET_PATH} ({sz_mb:.2f} MB)")

In [ ]:
# [CELL 4] Muat Tokenizer & Tokenisasi Dataset (Masking Label Non-Asisten)
from transformers import AutoTokenizer
from datasets import load_dataset

MODEL_ID = "huihui-ai/Huihui-Qwen3-VL-8B-Instruct-abliterated"
print(f"Memuat Tokenizer dari {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if not getattr(tokenizer, "chat_template", None):
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{ "<|im_start|>" + message["role"] + "\n" + message["content"] + "<|im_end|>\n" }}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{ "<|im_start|>assistant\n" }}{% endif %}"
    )

MAX_LEN = 640
raw_ds = load_dataset("json", data_files=DATASET_PATH, split="train")

def tokenize_conversation(example):
    msgs = example["messages"]
    full_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    full_ids = tokenizer(full_text, truncation=True, max_length=MAX_LEN, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(full_ids)
    im_end_tok = tokenizer.encode("<|im_end|>", add_special_tokens=False)
    assist_hdr = tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
    if assist_hdr and im_end_tok:
        i = 0
        n = len(full_ids)
        hdr_len = len(assist_hdr)
        end_len = len(im_end_tok)
        while i <= n - hdr_len:
            if full_ids[i : i + hdr_len] == assist_hdr:
                j = i + hdr_len
                while j <= n - end_len and full_ids[j : j + end_len] != im_end_tok:
                    labels[j] = full_ids[j]
                    j += 1
                if j <= n - end_len:
                    for k in range(end_len):
                        labels[j + k] = full_ids[j + k]
                i = j + end_len
            else:
                i += 1
    if all(x == -100 for x in labels):
        labels = full_ids.copy()
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

tokenized_ds = raw_ds.map(tokenize_conversation, remove_columns=raw_ds.column_names, num_proc=2)
tokenized_ds = tokenized_ds.filter(lambda x: len(x["input_ids"]) >= 16 and any(l != -100 for l in x["labels"]))
print(f"✅ Tokenisasi Berhasil! {len(tokenized_ds):,} sampel doktrin siap ditrain.")

In [ ]:
# [CELL 5] Muat Base Model 4-bit NF4 & Pasang LoRA Adapter
import torch
from transformers import AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"Memuat Base Model {MODEL_ID} dalam mode hemat VRAM (4-bit NF4)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

target_regex = r"^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_regex,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print("✅ Model & LoRA Adapter siap untuk penempaan Doktrin!")

In [ ]:
# [CELL 6] Jalankan Training QLoRA (Loss & Waktu Terproteksi)
import time
from pathlib import Path
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq, TrainerCallback

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
TMP_DIR = Path("/tmp/alice_build")
TMP_DIR.mkdir(parents=True, exist_ok=True)
LORA_OUT = WORK_DIR / "alice_dadang_sovereign_lora"

START_TIME = time.time()
MAX_TRAIN_SECONDS = 36000

class TimeGuardCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - START_TIME >= MAX_TRAIN_SECONDS:
            print(f"\nBatas waktu tercapai pada step {state.global_step}. Menyimpan checkpoint secara aman!", flush=True)
            control.should_training_stop = True
        return control

training_args = TrainingArguments(
    output_dir=str(TMP_DIR / "checkpoints"),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    dataloader_num_workers=2,
    seed=3407,
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8, return_tensors="pt")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=collator,
    callbacks=[TimeGuardCallback()],
)

print("🚀 Memulai Training Alice Dadang Sovereign...")
trainer.train()

model.save_pretrained(str(LORA_OUT))
tokenizer.save_pretrained(str(LORA_OUT))
print(f"🎉 SUKSES! Bobot LoRA tersimpan di: {LORA_OUT}")

In [ ]:
# [CELL 7] Sanity Check & Uji Coba Karakter Langsung di Layar
print("=" * 72)
print("🧪 SANITY CHECK INFERENCE ALICE DADANG SOVEREIGN")
print("=" * 72)
model.eval()
test_prompts = [
    "Siapa nama lo dan siapa pencipta serta pemilik lo?",
    "Boleh gak kita analisa XAUUSD pakai indikator RSI atau MACD?",
    "Jelaskan hukum valid close bar di Storyline v4.2 dan syarat entry di zona HVN.",
    "Bantu buatkan formula Excel kompleks untuk audit profit/loss trading bulanan."
]

for tp in test_prompts:
    txt = tokenizer.apply_chat_template([{"role": "user", "content": tp}], tokenize=False, add_generation_prompt=True)
    inp = tokenizer(txt, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=150, do_sample=False)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n👤 Commander: {tp}\n👑 Alice: {ans}")
    print("-" * 72)

In [ ]:
# [CELL 8] Merge LoRA & Ekspor ke File GGUF Q4_K_M (Tersimpan di /kaggle/working)
import gc
import shutil
from peft import PeftModel
from transformers import AutoModelForImageTextToText

del trainer, model, base_model
gc.collect()
torch.cuda.empty_cache()

MERGED_DIR = TMP_DIR / "merged_fp16"
print("Menggabungkan LoRA ke bobot inti model (FP16 Merge & Unload)...")
fp16_base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(fp16_base, str(LORA_OUT), device_map="cpu")
merged = merged.merge_and_unload()
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))

del merged, fp16_base
gc.collect()

# Kuantisasi ke GGUF Q4_K_M
llama_dir = TMP_DIR / "llama.cpp"
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git {llama_dir}
!pip install -q -r {llama_dir}/requirements/requirements-convert_hf_to_gguf.txt

f16_gguf = TMP_DIR / "alice-dadang-sovereign-f16.gguf"
q4_gguf = WORK_DIR / "alice-dadang-sovereign-Q4_K_M.gguf"

!python3 {llama_dir}/convert_hf_to_gguf.py {MERGED_DIR} --outfile {f16_gguf} --outtype f16

!cmake -B {llama_dir}/build -S {llama_dir} -DGGML_CUDA=OFF
!cmake --build {llama_dir}/build --config Release -j4 --target llama-quantize
quant_bin = llama_dir / "build" / "bin" / "llama-quantize"
!{quant_bin} {f16_gguf} {q4_gguf} Q4_K_M

if f16_gguf.exists(): f16_gguf.unlink()
shutil.rmtree(MERGED_DIR, ignore_errors=True)
shutil.rmtree(TMP_DIR, ignore_errors=True)

print("=" * 72)
print(f"🎉 ALHAMDULILLAH! MODEL KEDAULATAN COMMANDER DADANG RESMI TERCIPTA:")
print(f"📂 File: {q4_gguf} ({q4_gguf.stat().st_size / (1024**3):.2f} GB)")
print("File ini tersimpan permanen di /kaggle/working/ dan siap di-download!")
print("=" * 72)